In [21]:
import nashpy as nash
import numpy as np
from treys import Card, Evaluator, Deck
import itertools
import pandas as pd

deck = Deck()
board = [Card.new("8h"), Card.new("2h"), Card.new("6c"), Card.new("2d"), Card.new("5c")]
print(board)
player1_hand = deck.draw(2)
player2_hand = deck.draw(2)


mod1_deck = Deck().cards.copy()
mod2_deck = Deck().cards.copy()


for c in board:
    mod1_deck.remove(c)
for c in player1_hand:
    mod1_deck.remove(c)


for c in board:
    mod2_deck.remove(c)
for c in player1_hand:
    mod2_deck.remove(c)


possibilities_p2 = list(itertools.combinations(mod2_deck,2)) 
scores_p2 = []
evaluator = Evaluator()
for i in possibilities_p2:
    score2 = evaluator.evaluate(board,list(i))
    scores_p2.append(score2) 


score_ser_p2 = pd.Series(scores_p2)
buckets_p2 = pd.qcut(score_ser_p2, q=4, labels = [1,2,3,4])




possibilities_p1 = list(itertools.combinations(mod1_deck,2))
scores_p1 = []
for i in possibilities_p1:
    score1 = evaluator.evaluate(board, list(i))
    scores_p1.append(score1)


score_ser_p1 = pd.Series(scores_p1)  
buckets_p1 = pd.qcut(score_ser_p1, q=4, labels=[1, 2, 3, 4])


print("P1 bucket counts:\n", buckets_p1.value_counts().sort_index())
print("P2 bucket counts:\n", buckets_p2.value_counts().sort_index())

def input_strategy(player_name):
    strategy = {}
    print(f"\nEnter strategy for {player_name}")

    for b in [1,2,3,4]:
        action = input(f"Bucket {b} action (fold/call/bet): ").lower()
        strategy[b] = action

    return strategy
    
p1_strategy1 = input_strategy("Player 1 Strategy 1")
p1_strategy2 = input_strategy("Player 1 Strategy 2")

p2_strategy1 = input_strategy("Player 2 Strategy 1")
p2_strategy2 = input_strategy("Player 2 Strategy 2")

p1_strategies = [p1_strategy1, p1_strategy2]
p2_strategies = [p2_strategy1, p2_strategy2]

# p1_strategies = [
#     {1:"fold", 2:"call", 3:"bet", 4:"bet"},
#     {1:"fold", 2:"fold", 3:"call", 4:"bet"},
# ]


# p2_strategies = [
#     {1:"fold", 2:"fold", 3:"bet", 4:"bet"},
#     {1:"call", 2:"call", 3:"bet", 4:"bet"},
# ]


def payoff_single(hand_1, hand_2, act_1, act_2):
    s1 = evaluator.evaluate(board, list(hand_1))
    s2 = evaluator.evaluate(board, list(hand_2))


    if act_1 == "bet":
        if act_2 == "fold":
            return (1, 0)
        
        return ((2, 0) if (s1 < s2) else (0, 2))
    if act_1 in ("call"):
        if act_2 in ("call"):    
            return ((1, 0) if (s1 < s2) else (0, 1))
        if act_2 == "fold":     
            return ((1, 0) if (s1 < s2) else (0, 1))
        if act_2 == "bet":
            
            if act_1 == "fold":
                return (0, 1)
            else:  
                return ((2, 0) if (s1 < s2) else (0, 2))
    if act_1 == "fold":
        if act_2 == "bet":
            return (0, 1)
        
        return ((1, 0) if (s1 < s2) else (0, 1))


    
    raise ValueError(f"Unhandled combination: P1={act_1}, P2={act_2}")






def expected_payoff(p1_map, p2_map):
    total1 = 0.0
    total2 = 0.0
    N = len(possibilities_p1) * len(possibilities_p2)


    for i, h1 in enumerate(possibilities_p1):
        b1 = int(buckets_p1.iloc[i])    
        a1 = p1_map[b1]                 
        for j, h2 in enumerate(possibilities_p2):
            b2 = int(buckets_p2.iloc[j])
            a2 = p2_map[b2]
            u1, u2 = payoff_single(h1, h2, a1, a2)
            total1 += u1
            total2 += u2


    return total1 / N, total2 / N


A = np.zeros((len(p1_strategies), len(p2_strategies)))
B = np.zeros_like(A)


for i, p1_map in enumerate(p1_strategies):
    for j, p2_map in enumerate(p2_strategies):
        u1, u2 = expected_payoff(p1_map, p2_map)
        A[i, j] = u1
        B[i, j] = u2

p1_labels = [f"P1_Strategy_{i}" for i in range(len(p1_strategies))]
p2_labels = [f"P2_Strategy_{j}" for j in range(len(p2_strategies))]

print("=== P1 payoff matrix A ===")
print(pd.DataFrame(A, index=p1_labels, columns=p2_labels))

print("\n=== P2 payoff matrix B ===")
print(pd.DataFrame(B, index=p1_labels, columns=p2_labels))
print()


game = nash.Game(A, B)
equilibria = list(game.support_enumeration())


print("Nash equilibria (σ_P1, σ_P2):")
for idx, (σ1, σ2) in enumerate(equilibria, start=1):
    print(f"Equilibrium {idx}:")
    print("  P1 mix =", σ1)
    print("  P2 mix =", σ2)
    print()

[4204049, 73730, 1082379, 81922, 557831]
P1 bucket counts:
 1    256
2    250
3    248
4    236
Name: count, dtype: int64
P2 bucket counts:
 1    256
2    250
3    248
4    236
Name: count, dtype: int64

Enter strategy for Player 1 Strategy 1


Bucket 1 action (fold/call/bet):  fold
Bucket 2 action (fold/call/bet):  call
Bucket 3 action (fold/call/bet):  bet
Bucket 4 action (fold/call/bet):  bet



Enter strategy for Player 1 Strategy 2


Bucket 1 action (fold/call/bet):  fold
Bucket 2 action (fold/call/bet):  fold
Bucket 3 action (fold/call/bet):  call
Bucket 4 action (fold/call/bet):  bet



Enter strategy for Player 2 Strategy 1


Bucket 1 action (fold/call/bet):  fold
Bucket 2 action (fold/call/bet):  fold
Bucket 3 action (fold/call/bet):  fold
Bucket 4 action (fold/call/bet):  fold



Enter strategy for Player 2 Strategy 2


Bucket 1 action (fold/call/bet):  call
Bucket 2 action (fold/call/bet):  call
Bucket 3 action (fold/call/bet):  call
Bucket 4 action (fold/call/bet):  call


=== P1 payoff matrix A ===
               P2_Strategy_0  P2_Strategy_1
P1_Strategy_0       0.865983       0.606197
P1_Strategy_1       0.704204       0.517472

=== P2 payoff matrix B ===
               P2_Strategy_0  P2_Strategy_1
P1_Strategy_0       0.134017       0.882692
P1_Strategy_1       0.295796       0.720912

Nash equilibria (σ_P1, σ_P2):
Equilibrium 1:
  P1 mix = [1. 0.]
  P2 mix = [0. 1.]

